# GradCAM vs BBox Segmentation - Google Colab Setup

本notebook用于在Google Colab中复现弱监督分割项目的所有实验。

## 主要改进 (2024更新版):
- ✅ 所有模块添加验证集支持 (train/val/test 三分)
- ✅ 数据划分改为标准的70/15/15 (合并trainval和test后重新划分)
- ✅ 训练轮数增加到10 epochs，带早停机制
- ✅ 修复loss计算，显示平均loss
- ✅ 统一结果输出和对比功能
- ✅ 新增快速测试模式 (使用10%数据快速验证流程)

## Setup步骤:
1. 上传本notebook到Google Colab
2. 启用GPU加速: Runtime → Change runtime type → GPU
3. 上传项目文件或从GitHub克隆
4. **首次运行建议：使用快速测试模式 (10%数据) 验证流程**
5. 确认无误后，使用完整数据集重新训练

## 测试模式选择:
- ⚡ **极速测试 (推荐)**: `--data_percentage 0.01`，使用1%数据，**< 10分钟**完成全部实验
- 🚀 **快速测试**: `--data_percentage 0.1`，使用10%数据，约30-40分钟
- 📊 **完整训练**: 不添加参数或 `--data_percentage 1.0`，使用全部数据，约60-90分钟


## 第1步: 安装依赖包


In [ ]:
# 安装核心依赖
!pip install torch torchvision --index-url https://download.pytorch.org/whl/cu121
!pip install opencv-python matplotlib pillow
!pip install git+https://github.com/lucasb-eyer/pydensecrf.git

print("\n✅ 所有依赖包安装完成！")


## 第2步: 验证GPU可用性


In [ ]:
import torch
import sys

print("="*60)
print("系统配置检查")
print("="*60)
print(f"Python版本: {sys.version.split()[0]}")
print(f"PyTorch版本: {torch.__version__}")
print(f"CUDA可用: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"CUDA设备: {torch.cuda.get_device_name(0)}")
    print(f"CUDA版本: {torch.version.cuda}")
    print(f"显存大小: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")
    print("\n✅ GPU已启用 - 训练将使用GPU加速")
else:
    print("\n⚠️  GPU未启用 - 训练将在CPU上进行（速度较慢）")
    print("   建议: Runtime → Change runtime type → GPU")


## 第3步: 上传项目文件

**选项A: 从GitHub克隆 (推荐)**
- 取消注释下方代码并修改URL

**选项B: 手动上传**
- 点击左侧文件夹图标
- 上传所有项目文件到 `/content/` 目录


In [ ]:
# 选项A: 从GitHub克隆 (取消注释并修改URL)
# !git clone https://github.com/KarenShark/EE4211-Project.git
# %cd EE4211-Project

# 如果已经克隆，直接切换到项目目录
import os
if os.path.exists('/content/EE4211-Project'):
    %cd /content/EE4211-Project
    print("✅ 已切换到项目目录")
else:
    print("⚠️  项目目录不存在，请先上传项目文件或克隆仓库")


## 第4步: 验证项目文件结构


In [ ]:
import os
import sys

# 确保工作目录正确
if not os.path.exists('data.py'):
    possible_dirs = ['/content', '/content/EE4211-Project']
    for d in possible_dirs:
        if os.path.exists(os.path.join(d, 'data.py')):
            os.chdir(d)
            break

# 添加项目根目录到Python路径
project_root = os.getcwd()
if project_root not in sys.path:
    sys.path.insert(0, project_root)

print(f"当前工作目录: {project_root}")
print("\n检查项目文件结构:")
print("="*60)

required_dirs = ['weakly-supervised', 'fully-supervised', 'open-ended-question', 'utils']
required_files = ['data.py', 'ground_truth.py', 'compare_results.py']

all_found = True
for dir_name in required_dirs:
    if os.path.exists(dir_name) and os.path.isdir(dir_name):
        print(f"✅ 目录存在: {dir_name}")
    else:
        print(f"❌ 目录缺失: {dir_name}")
        all_found = False

for file_name in required_files:
    if os.path.exists(file_name):
        print(f"✅ 文件存在: {file_name}")
    else:
        print(f"❌ 文件缺失: {file_name}")
        all_found = False

print("="*60)
if all_found:
    print("\n✅ 所有必需文件和目录都已找到！可以开始运行实验。")
else:
    print("\n⚠️  缺少某些文件或目录，请上传完整的项目文件！")


## 第5步: 弱监督分割 (主要方法 - 使用CRF)

这是论文的主要方法，使用CAM生成的弱标签，配合CRF后处理。

**改进点:**
- ✅ 训练集划分为train/val (70/15)
- ✅ 训练轮数增加到10 epochs
- ✅ 早停机制 (patience=3)
- ✅ 修复loss计算


In [ ]:
print("="*80)
print("开始训练: 弱监督分割 (CAM + CRF)")
print("="*80)

# ⚡ 极速测试模式 (1%数据，约2-3分钟) - 首次运行强烈推荐！
!python weakly-supervised/main.py --use_crf True --foreground_threshold 0.05 --data_percentage 0.01

# 🚀 快速测试模式 (10%数据，约10-15分钟) - 取消下方注释使用
# !python weakly-supervised/main.py --use_crf True --foreground_threshold 0.05 --data_percentage 0.1

# 📊 完整训练模式 (100%数据，约20-30分钟) - 取消下方注释使用
# !python weakly-supervised/main.py --use_crf True --foreground_threshold 0.05


## 第6步: 全监督基线 (DeepLabV3+)

使用全监督的DeepLabV3+作为性能上限基准。

**改进点:**
- ✅ 训练集划分为train/val/test (70/15/15)
- ✅ 训练轮数增加到10 epochs with early stopping
- ✅ 修复loss计算


In [ ]:
print("="*80)
print("开始训练: 全监督DeepLabV3+ (基线)")
print("="*80)

# ⚡ 极速测试模式 (1%数据，约1-2分钟) - 首次运行强烈推荐！
!python fully-supervised/main.py --model_name deeplab --data_percentage 0.01

# 🚀 快速测试模式 (10%数据，约5-8分钟) - 取消下方注释使用
# !python fully-supervised/main.py --model_name deeplab --data_percentage 0.1

# 📊 完整训练模式 (100%数据，约15-25分钟) - 取消下方注释使用
# !python fully-supervised/main.py --model_name deeplab


## 第7步: 开放性问题 (边界框 vs CAM)

比较两种弱监督标注方法：边界框+GrabCut vs CAM。


In [ ]:
print("="*80)
print("开始训练: 边界框弱监督分割")
print("="*80)

# ⚡ 极速测试模式 (1%数据，约1-2分钟) - 首次运行强烈推荐！
!python open-ended-question/main.py --data_percentage 0.01

# 🚀 快速测试模式 (10%数据，约5-8分钟) - 取消下方注释使用
# !python open-ended-question/main.py --data_percentage 0.1

# 📊 完整训练模式 (100%数据，约15-20分钟) - 取消下方注释使用
# !python open-ended-question/main.py


## 第8步: 消融实验 - 无CRF对比


In [ ]:
print("="*80)
print("消融实验: 弱监督分割 (无CRF)")
print("="*80)

# ⚡ 极速测试模式 (1%数据，约2-3分钟) - 首次运行强烈推荐！
!python weakly-supervised/main.py --use_crf False --foreground_threshold 0.05 --data_percentage 0.01

# 🚀 快速测试模式 (10%数据，约10-15分钟) - 取消下方注释使用
# !python weakly-supervised/main.py --use_crf False --foreground_threshold 0.05 --data_percentage 0.1

# 📊 完整训练模式 (100%数据，约20-30分钟) - 取消下方注释使用
# !python weakly-supervised/main.py --use_crf False --foreground_threshold 0.05


## 第9步: 生成统一的结果对比报告 ✨ 新功能

使用统一的结果管理系统收集和对比所有实验结果。


In [ ]:
print("="*80)
print("生成统一的结果对比报告")
print("="*80)

!python compare_results.py

print("\n" + "="*80)
print("结果已保存到 output/ 目录")
print("  - output/comparisons/comparison_report.txt")
print("  - output/comparisons/results_comparison.csv")
print("  - output/visualizations/")
print("="*80)


## 第10步: 查看对比报告


In [ ]:
import os

report_path = 'output/comparisons/comparison_report.txt'

if os.path.exists(report_path):
    with open(report_path, 'r') as f:
        print(f.read())
else:
    print(f"⚠️  报告文件不存在: {report_path}")


## 第11步: 查看CSV对比表格


In [ ]:
import pandas as pd
import os

csv_path = 'output/comparisons/results_comparison.csv'

if os.path.exists(csv_path):
    df = pd.read_csv(csv_path)
    print("="*80)
    print("所有实验结果对比表格")
    print("="*80)
    display(df)
    
    # 下载CSV文件
    from google.colab import files
    files.download(csv_path)
else:
    print(f"⚠️  CSV文件不存在: {csv_path}")


## 可选: 保存结果到Google Drive


In [ ]:
# 挂载Google Drive
from google.colab import drive
drive.mount('/content/drive')

# 复制结果到Drive
import shutil
drive_dir = '/content/drive/MyDrive/EE4211-Project-Results'
os.makedirs(drive_dir, exist_ok=True)

if os.path.exists('output'):
    shutil.copytree('output', os.path.join(drive_dir, 'output'), dirs_exist_ok=True)
    print(f"✅ 结果已保存到: {drive_dir}")


## 重要提示

### GPU配置
- 启用GPU: **Runtime → Change runtime type → GPU**
- 推荐T4或更好的GPU

### 改进总结
本次更新的主要改进:
1. ✅ 所有模块添加train/val/test三分
2. ✅ 数据划分改为标准的70/15/15 (合并trainval和test后重新划分)
3. ✅ 训练轮数增加到10 epochs with early stopping
4. ✅ 修复loss计算，显示平均loss
5. ✅ 统一结果输出和对比工具
6. ✅ 新增快速测试模式 (--data_percentage参数)

### 预期训练时间 (T4 GPU)
**⚡ 极速测试模式 (1%数据) - 强烈推荐首次运行**:
- 弱监督分割 (CAM+CRF): ~2-3分钟
- 全监督DeepLabV3+: ~1-2分钟
- 边界框弱监督: ~1-2分钟
- 消融实验 (无CRF): ~2-3分钟
- **总计: < 10分钟 ⚡**

**🚀 快速测试模式 (10%数据)**:
- 弱监督分割 (CAM+CRF): ~10-15分钟
- 全监督DeepLabV3+: ~5-8分钟
- 边界框弱监督: ~5-8分钟
- 消融实验 (无CRF): ~10-15分钟
- **总计: ~30-45分钟**

**📊 完整训练模式 (100%数据)**:
- 弱监督分割 (CAM+CRF): ~20-30分钟
- 全监督DeepLabV3+: ~15-25分钟
- 边界框弱监督: ~15-20分钟
- 消融实验 (无CRF): ~20-30分钟
- **总计: ~70-105分钟**

### 使用建议 🎯
1. ⚡ **首次运行**: 使用极速测试模式（1%数据）快速验证流程 **< 10分钟**
2. ✅ **确认无误**: 注释掉极速测试，取消注释完整训练命令
3. 📊 **获得最终结果**: 重新运行所有训练cells

### 结果位置
所有结果统一保存在 `output/` 目录:
- `output/comparisons/` - 对比报告和表格
- `output/visualizations/` - 预测结果可视化
